# 💍 Wedding Planner — Module 2 Capstone Exercise

This notebook consolidates everything from lessons 2.1–2.3:

| Concept | Where it's used |
|---|---|
| 🔌 MCP (stdio) | Venue agent uses Tavily search via local MCP server |
| 🌐 MCP (streamable_http) | Transport agent uses Kiwi flights via remote MCP |
| 📐 Runtime context (`context_schema`) | All agents declare `WeddingContext` as their schema |
| 📦 Runtime context (`context=`) | Passed at invoke time with real couple data |
| 🗂️ State (`state_schema`) | `WeddingPlanState` accumulates results across subagents |
| ✏️ State write | DJ agent writes playlist via `Command(update={...})` |
| 👁️ State read | Coordinator reads progress via `runtime.state` |
| 🤝 Multi-agent | Coordinator delegates to 4 specialist subagents |
| 💾 Checkpointer | `InMemorySaver` enables multi-turn conversation |

---

## 🗂️ File structure

```
2.4_5_wedding_exercisee/
├── context_and_state.py   # WeddingContext (dataclass) + WeddingPlanState (AgentState)
├── mcp_servers.py         # FastMCP server (stdio) + get_mcp_tools() helper
├── subagents.py           # 4 specialist agents: venue, catering, transport, DJ
├── wedding_planner.py     # Coordinator + main entry point
└── wedding_planner.ipynb  # This notebook
```

In [ ]:
import sys
import os
import asyncio

# LESSON (from 2.1_mcp.ipynb): Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

# Make sure imports from this folder work
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from dotenv import load_dotenv
load_dotenv('../../../.env')  # load OPENAI_API_KEY, TAVILY_API_KEY

## Step 1 — Context & State schemas

Before any agents are created, we define:
- `WeddingContext` — a dataclass with the couple's details (read-only, injected at invoke time)
- `WeddingPlanState` — extends `AgentState` with fields the coordinator fills in as subagents report back

In [ ]:
from context_and_state import WeddingContext, WeddingPlanState

# This is the runtime context — the actual data we'll pass at invoke time
wedding_context = WeddingContext(
    couple_names="Alice & Bob",
    wedding_date="2026-09-15",
    wedding_location="Paris, France",
    guest_count=80,
    budget_usd=50_000,
    music_style="jazz and soul",
    dietary_restrictions="vegan options required",
    home_city="New York",
)

print(wedding_context)

## Step 2 — Connect to MCP servers

We connect to two MCP servers:
1. **Local (stdio)** — `mcp_servers.py` runs as a subprocess, exposes `search_web` via Tavily
2. **Remote (streamable_http)** — Kiwi.com MCP server, exposes `search-flight`

`MultiServerMCPClient` handles both transports and returns a unified tool list.

In [ ]:
from mcp_servers import get_mcp_tools

mcp_tools, mcp_client = await get_mcp_tools()

print("MCP tools available:")
for t in mcp_tools:
    print(f"  - {t.name}: {t.description[:60]}...")

### Bonus: MCP Resources & Prompts

LESSON (from 2.1_mcp.ipynb): MCP servers expose more than just tools.
- `get_resources()` — static data the server provides
- `get_prompt()` — reusable prompt templates defined by the server

This is how the 2.1_mcp course loaded the system prompt from the MCP server.

In [ ]:
from mcp_servers import get_mcp_resources_and_prompts

resources, prompt_text = await get_mcp_resources_and_prompts(mcp_client)

print("Resources:", resources)
print("\nPrompt from MCP server:")
print(prompt_text)

## Step 3 — Create the 4 specialist subagents

Each subagent is a standalone `create_agent()` with:
- Its own tools (some from MCP, some regular `@tool`)
- `context_schema=WeddingContext` so it knows the shape of context it'll receive
- A focused `system_prompt` for its domain

In [ ]:
from subagents import (
    create_venue_agent,
    create_catering_agent,
    create_transport_agent,
    create_dj_agent,
)

search_tools = [t for t in mcp_tools if t.name == "search_web"]
flight_tools = [t for t in mcp_tools if "flight" in t.name.lower()]

venue_agent     = create_venue_agent(search_tools)
catering_agent  = create_catering_agent()
transport_agent = create_transport_agent(flight_tools)
dj_agent        = create_dj_agent()

print("✅ All 4 subagents created")

## Step 4 — Test a subagent directly

Before running the full coordinator, let's test the catering agent in isolation.

Notice how we pass `context=wedding_context` at invoke time — this is the actual data.
The agent reads it via `ToolRuntime` inside its tools.

In [ ]:
from langchain.messages import HumanMessage
from pprint import pprint

response = catering_agent.invoke(
    {"messages": [HumanMessage(content="Plan catering for our wedding")]},
    context=wedding_context,   # ← runtime context injected here
)

print(response["messages"][-1].content)

## Step 5 — Test the DJ agent (state write demo)

The DJ agent is special: it writes the playlist into `WeddingPlanState` via `Command(update={...})`.

After the call, you can see `playlist_result` in the response dict — that's the state field.

In [ ]:
dj_response = dj_agent.invoke(
    {"messages": [HumanMessage(content="Create a wedding playlist for us")]},
    {"configurable": {"thread_id": "dj-test-1"}},
    context=wedding_context,
)

print("Last message:", dj_response["messages"][-1].content)
print("\nState field 'playlist_result':")
print(dj_response.get("playlist_result", "(not set)"))

### Bonus: Pass state fields directly in invoke dict

LESSON (from 2.2_state.ipynb): You can also pass initial state values directly
in the invoke dict alongside messages. This is how the course set `favourite_colour`
without the agent needing to discover it via a tool call first.

```python
# From the course:
response = agent.invoke(
    {"messages": [...], "favourite_colour": "green"},
    {"configurable": {"thread_id": "10"}}
)
```

In [ ]:
# Pass a pre-set playlist_result directly in the invoke dict
# This seeds the state without the agent needing to call save_playlist_to_state
dj_response_seeded = dj_agent.invoke(
    {
        "messages": [HumanMessage(content="What playlist do I have?")],
        "playlist_result": "Pre-seeded: At Last by Etta James, Fly Me to the Moon by Sinatra"
    },
    {"configurable": {"thread_id": "dj-test-seeded"}},
    context=wedding_context,
)

print("Agent response:", dj_response_seeded["messages"][-1].content)
print("\nState playlist_result:", dj_response_seeded.get("playlist_result", "(not set)"))

## Step 6 — Run the full coordinator

Now we run the complete multi-agent system.

The coordinator will:
1. Call all 4 subagents via wrapper tools
2. Each result gets written into `WeddingPlanState`
3. The coordinator reads progress via `read_planning_progress`
4. Writes the final assembled plan via `write_final_plan`

⚠️ This makes real API calls (Tavily + Kiwi + OpenAI). It may take 30–60 seconds.

In [ ]:
from wedding_planner import build_coordinator

coordinator, _ = await build_coordinator(wedding_context)

config = {"configurable": {"thread_id": "wedding-plan-notebook-1"}}

user_request = (
    f"Please plan our complete wedding! "
    f"We are {wedding_context.couple_names}, getting married on {wedding_context.wedding_date} "
    f"in {wedding_context.wedding_location}. "
    f"We have {wedding_context.guest_count} guests and a budget of ${wedding_context.budget_usd:,}. "
    f"We love {wedding_context.music_style} music and need {wedding_context.dietary_restrictions}. "
    f"We're flying from {wedding_context.home_city}. Please coordinate everything!"
)

response = await coordinator.ainvoke(
    {"messages": [HumanMessage(content=user_request)]},
    config=config,
    context=wedding_context,
)

print(response["messages"][-1].content)

## Step 7 — Inspect the accumulated state

After the coordinator finishes, all subagent results are in state.
This is the power of `WeddingPlanState` — nothing is lost between turns.

In [ ]:
print("📊 State after full planning run:")
print("-" * 50)
for field in ["venue_result", "catering_result", "transport_result", "playlist_result", "final_plan"]:
    value = response.get(field, "")
    if value:
        print(f"\n✅ {field}:")
        print(value[:200] + "..." if len(value) > 200 else value)
    else:
        print(f"\n⏳ {field}: (not set)")

## Step 8 — Multi-turn follow-up (checkpointer demo)

Because we used `InMemorySaver` with the same `thread_id`, the coordinator
remembers the entire conversation. We can ask follow-up questions without
re-running all the subagents.

In [ ]:
follow_up = "Can you remind me just the playlist and the venue recommendation?"

follow_up_response = await coordinator.ainvoke(
    {"messages": [HumanMessage(content=follow_up)]},
    config=config,   # same thread_id — coordinator remembers everything
    context=wedding_context,
)

print(follow_up_response["messages"][-1].content)